In [1]:
!pip install xgboost tensorflow ipywidgets -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from google.colab import files

from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries loaded!")

display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@700;900&family=Rajdhani:wght@400;600&display=swap');
  .banner {
    background: linear-gradient(135deg, #0f0c29, #302b63, #24243e);
    border-radius: 16px; padding: 40px 30px; text-align: center;
    border: 1px solid rgba(100,120,255,0.3);
    box-shadow: 0 0 40px rgba(100,120,255,0.2), inset 0 0 60px rgba(0,0,0,0.4);
    margin-bottom: 10px;
  }
  .banner h1 {
    font-family: 'Orbitron', monospace; font-size: 2em; font-weight: 900;
    background: linear-gradient(90deg, #a78bfa, #60a5fa, #34d399);
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    margin: 0 0 8px 0; letter-spacing: 2px;
  }
  .banner p { font-family: 'Rajdhani', sans-serif; color: #94a3b8; font-size: 1.1em; margin: 0; }
  .step-row { display: flex; justify-content: center; gap: 12px; margin-top: 20px; flex-wrap: wrap; }
  .step-card {
    background: rgba(255,255,255,0.05); border: 1px solid rgba(255,255,255,0.1);
    border-radius: 10px; padding: 10px 18px;
    font-family: 'Rajdhani', sans-serif; color: #cbd5e1; font-size: 0.9em;
  }
  .step-card span { color: #60a5fa; font-weight: 700; margin-right: 6px; }
  .model-row { display: flex; justify-content: center; gap: 10px; margin-top: 16px; flex-wrap: wrap; }
  .model-badge {
    background: rgba(167,139,250,0.1); border: 1px solid rgba(167,139,250,0.3);
    border-radius: 8px; padding: 6px 14px;
    font-family: 'Rajdhani', sans-serif; color: #a78bfa; font-size: 0.82em;
    letter-spacing: 0.5px;
  }
</style>
<div class="banner">
  <h1>🎓 Student Performance Predictor</h1>
  <p>Advanced ML + Deep Learning Academic Analytics System</p>
  <div class="model-row">
    <div class="model-badge">🚀 Multi-Output GBR</div>
    <div class="model-badge">🧠 Hybrid RF + Logistic</div>
    <div class="model-badge">📈 Temporal Progression</div>
    <div class="model-badge">🔬 Autoencoder DL</div>
    <div class="model-badge">⚡ Ensemble</div>
  </div>
  <div class="step-row">
    <div class="step-card"><span>①</span>Upload</div>
    <div class="step-card"><span>②</span>Train</div>
    <div class="step-card"><span>③</span>Predict</div>
    <div class="step-card"><span>④</span>Dashboard</div>
    <div class="step-card"><span>⑤</span>Download</div>
  </div>
</div>
"""))

display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@700&family=Rajdhani:wght@400;600&display=swap');
  .section-title {
    font-family: 'Orbitron', monospace;
    background: linear-gradient(135deg, #0f0c29, #302b63);
    color: #a78bfa; padding: 14px 22px; border-radius: 10px;
    font-size: 0.9em; border-left: 4px solid #a78bfa;
    margin: 10px 0; letter-spacing: 1px;
  }
</style>
<div class="section-title">📂 STEP 1 — Upload Your Data Files</div>
"""))

upload_btn1 = widgets.Button(description='📁 Upload sem1.csv',
    style={'button_color': '#4f46e5'},
    layout=widgets.Layout(width='220px', height='42px'))
upload_btn2 = widgets.Button(description='📁 Upload sem 2.csv',
    style={'button_color': '#0891b2'},
    layout=widgets.Layout(width='220px', height='42px'))
status1 = widgets.Label(value='⏳ Not uploaded')
status2 = widgets.Label(value='⏳ Not uploaded')

sem1, sem2, sem2_original = None, None, None

def upload_sem1(b):
    global sem1
    print("📂 Choose sem1.csv ↓")
    files.upload()
    sem1 = pd.read_csv("sem1.csv")
    status1.value = f'✅ sem1.csv — {sem1.shape[0]} students, {sem1.shape[1]} cols'

def upload_sem2(b):
    global sem2, sem2_original
    print("📂 Choose sem 2.csv ↓")
    files.upload()
    sem2 = pd.read_csv("sem 2.csv")
    sem2_original = sem2.copy()
    status2.value = f'✅ sem 2.csv — {sem2.shape[0]} students, {sem2.shape[1]} cols'

upload_btn1.on_click(upload_sem1)
upload_btn2.on_click(upload_sem2)
display(widgets.HBox([upload_btn1, status1]))
display(widgets.HBox([upload_btn2, status2]))

sem2_subjects = [
    "Degn & Analysis of Algorithms (30)",
    "Artificial Intelligence (30)",
    "Computer Networks - Theory (30)",
    "Software Engineering Methodologies - Theory (30)"
]
lab_subjects = [
    "Computer Networks - Lab (15)",
    "Software Engineering Methodologies - Lab (15)",
    "Web Technologies - Lab (15)",
    "Python Lab (15)"
]
sem1_pairs = [
    ("Operating System mid term cia(30)",              "Operating System Final (70)"),
    ("Data Structures and Algorithms MidSem cia (30)", "Data Structures and Algorithms Final (70)"),
    ("maths  MidSem (30)",                             "maths Final (70)"),
    ("DBMS MidSem cia (30)",                           "DBMS Final (70)"),
    ("DS PRACTICAL(15)",                               "DS PRACTICAL FINAL(35)"),
    ("DBMS PRACTICAL (15)",                            "DBMS PRACTICAL FINAL (35)"),
    ("JAVA PRACTICAL (15)",                            "JAVA PRACTICAL FINAL (35)"),
    ("Problem Solving Techniques PRACTICAL (15)",      "Problem Solving Techniques FINAL (35)"),
    ("Java MidSem (15)",                               "Java final (35)")
]

def clean_marks(df):
    df = df.copy()
    for col in df.columns:
        df[col] = df[col].astype(str).str.replace("+","",regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def grade(c):
    if c >= 9:   return "O"
    elif c >= 8: return "A+"
    elif c >= 7: return "A"
    elif c >= 6: return "B+"
    elif c >= 5: return "B"
    else:        return "F"

print("✅ Subjects and helper functions defined!")

display(HTML('<div class="section-title">⚙️ STEP 2 — Train All 4 Advanced Models</div>'))

train_btn = widgets.Button(description='🚀 Train All Models',
    style={'button_color': '#059669'},
    layout=widgets.Layout(width='220px', height='44px'))
train_out = widgets.Output()

# Global model storage
M1_model = None   # Multi-Output GBR
M2_rf    = None   # Hybrid RF
M2_lr    = None   # Logistic Regression
M3_model = None   # Temporal GBR
M4_enc   = None   # Autoencoder encoder
M4_dec   = None   # Autoencoder decoder
M4_scaler= None
model_maes = {}

def run_training(b):
    global sem1, sem2, sem2_original
    global M1_model, M2_rf, M2_lr, M3_model, M4_enc, M4_dec, M4_scaler, model_maes

    with train_out:
        clear_output()
        if sem1 is None or sem2 is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Upload both files first!</p>'))
            return

        # ── Clean ──
        sem1_c = clean_marks(sem1).fillna(0)
        sem2_c = clean_marks(sem2).fillna(0)
        sem2_original = sem2_c.copy()

        sem1_inputs  = [p[0] for p in sem1_pairs]
        sem1_outputs = [p[1] for p in sem1_pairs]

        X_single = sem1_c[sem1_inputs[0]].values.reshape(-1,1)
        y_single = sem1_c[sem1_outputs[0]].values
        X_tr, X_te, y_tr, y_te = train_test_split(X_single, y_single, test_size=0.2, random_state=42)

        # ════════════════════════════════════════════
        # MODEL 1 — Performance-Aware Multi-Output GBR
        # ════════════════════════════════════════════
        display(HTML('<p style="color:#60a5fa;font-family:monospace;font-weight:bold">🚀 Training Model 1: Multi-Output GBR...</p>'))
        X_multi = sem1_c[sem1_inputs].values
        y_multi = sem1_c[sem1_outputs].values
        X_tr1, X_te1, y_tr1, y_te1 = train_test_split(X_multi, y_multi, test_size=0.2, random_state=42)
        M1_model = MultiOutputRegressor(GradientBoostingRegressor(n_estimators=100, random_state=42))
        M1_model.fit(X_tr1, y_tr1)
        m1_preds = M1_model.predict(X_te1)
        m1_mae   = mean_absolute_error(y_te1, m1_preds)
        model_maes["M1_MultiOutput_GBR"] = m1_mae
        display(HTML(f'<p style="color:#34d399;font-family:monospace">✅ Model 1 done — MAE: {m1_mae:.2f}</p>'))

        # ════════════════════════════════════════════
        # MODEL 2 — Knowledge-Aware Hybrid RF + Logistic
        # ════════════════════════════════════════════
        display(HTML('<p style="color:#60a5fa;font-family:monospace;font-weight:bold">🧠 Training Model 2: Hybrid RF + Logistic...</p>'))
        train_list = []
        for inp, out in sem1_pairs:
            temp = pd.DataFrame({"Internal": sem1_c[inp], "Final": sem1_c[out]})
            train_list.append(temp)
        train_df = pd.concat(train_list, ignore_index=True)
        X2 = train_df[["Internal"]]; y2 = train_df["Final"]
        X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X2, y2, test_size=0.2, random_state=42)
        M2_rf = RandomForestRegressor(n_estimators=150, random_state=42)
        M2_rf.fit(X_tr2, y_tr2)
        rf_preds = M2_rf.predict(X_te2)
        m2_mae   = mean_absolute_error(y_te2, rf_preds)
        model_maes["M2_Hybrid_RF"] = m2_mae

        # Logistic for Pass/Fail refinement
        sem1_c["__pass__"] = (sem1_c.get("Result", "Pass") == "Pass").astype(int) if "Result" in sem1_c.columns else 1
        X_lr = sem1_c[sem1_inputs].fillna(0)
        y_lr = sem1_c["__pass__"]
        if y_lr.nunique() > 1:
            M2_lr = LogisticRegression(max_iter=1000)
            M2_lr.fit(X_lr, y_lr)
        display(HTML(f'<p style="color:#34d399;font-family:monospace">✅ Model 2 done — MAE: {m2_mae:.2f}</p>'))

        # ════════════════════════════════════════════
        # MODEL 3 — Temporal Student Progression Model
        # ════════════════════════════════════════════
        display(HTML('<p style="color:#60a5fa;font-family:monospace;font-weight:bold">📈 Training Model 3: Temporal Progression...</p>'))

        # Calculate performance delta (progress from internal → final in sem1)
        delta_list = []
        for inp, out in sem1_pairs:
            delta = sem1_c[out] - sem1_c[inp]
            delta_list.append(delta)
        delta_df = pd.concat(delta_list, ignore_index=True)
        internal_all = pd.concat([sem1_c[inp] for inp, _ in sem1_pairs], ignore_index=True)

        X3 = pd.DataFrame({"Internal": internal_all, "Delta": delta_df})
        y3 = pd.concat([sem1_c[out] for _, out in sem1_pairs], ignore_index=True)
        X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X3, y3, test_size=0.2, random_state=42)
        M3_model = GradientBoostingRegressor(n_estimators=150, learning_rate=0.05, random_state=42)
        M3_model.fit(X_tr3, y_tr3)
        m3_mae = mean_absolute_error(y_te3, M3_model.predict(X_te3))
        model_maes["M3_Temporal_GBR"] = m3_mae
        display(HTML(f'<p style="color:#34d399;font-family:monospace">✅ Model 3 done — MAE: {m3_mae:.2f}</p>'))

        # ════════════════════════════════════════════
        # MODEL 4 — Autoencoder Deep Learning
        # ════════════════════════════════════════════
        display(HTML('<p style="color:#60a5fa;font-family:monospace;font-weight:bold">🔬 Training Model 4: Autoencoder (Deep Learning)...</p>'))

        scaler = MinMaxScaler()
        X_ae   = sem1_c[sem1_inputs + sem1_outputs].fillna(0).values
        X_ae_scaled = scaler.fit_transform(X_ae)
        M4_scaler = scaler

        input_dim = X_ae_scaled.shape[1]
        inp_layer = Input(shape=(input_dim,))
        encoded   = Dense(16, activation='relu')(inp_layer)
        encoded   = Dense(8,  activation='relu')(encoded)
        latent    = Dense(4,  activation='relu')(encoded)
        decoded   = Dense(8,  activation='relu')(latent)
        decoded   = Dense(16, activation='relu')(decoded)
        out_layer = Dense(input_dim, activation='sigmoid')(decoded)

        autoencoder = Model(inp_layer, out_layer)
        encoder_model = Model(inp_layer, latent)
        autoencoder.compile(optimizer='adam', loss='mse')

        autoencoder.fit(X_ae_scaled, X_ae_scaled,
                       epochs=80, batch_size=8, verbose=0,
                       callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])

        reconstructed = autoencoder.predict(X_ae_scaled, verbose=0)
        reconstructed_orig = scaler.inverse_transform(reconstructed)
        m4_mae = mean_absolute_error(X_ae, reconstructed_orig)
        model_maes["M4_Autoencoder"] = m4_mae
        M4_enc = encoder_model
        M4_dec = autoencoder
        display(HTML(f'<p style="color:#34d399;font-family:monospace">✅ Model 4 done — Reconstruction MAE: {m4_mae:.2f}</p>'))

        # ── Accuracy Chart ──
        names  = list(model_maes.keys())
        maes   = list(model_maes.values())
        accs   = [(1 - m/70)*100 for m in maes]

        fig, ax = plt.subplots(figsize=(9,4))
        fig.patch.set_facecolor('#0f172a'); ax.set_facecolor('#1e293b')
        colors = ["#a78bfa","#60a5fa","#34d399","#fb923c"]
        bars = ax.bar(names, accs, color=colors, width=0.5, edgecolor='none')
        for bar in bars:
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                    f'{bar.get_height():.1f}%', ha='center', color='white', fontsize=9, fontweight='bold')
        ax.set_title("All 4 Model Accuracy Comparison", color='white', fontsize=13, fontweight='bold', pad=12)
        ax.set_ylabel("Accuracy (%)", color='#94a3b8')
        ax.tick_params(colors='#94a3b8', rotation=15, labelsize=8)
        ax.spines[:].set_visible(False)
        ax.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
        plt.tight_layout(); plt.show()

        display(HTML('<p style="color:#a78bfa;font-family:monospace;font-size:1.1em">🎯 All 4 models trained successfully!</p>'))

train_btn.on_click(run_training)
display(train_btn, train_out)



display(HTML('<div class="section-title">🔮 STEP 3 — Predict Semester 2 + Ensemble</div>'))

predict_btn = widgets.Button(description='⚡ Run All Predictions',
    style={'button_color': '#7c3aed'},
    layout=widgets.Layout(width='230px', height='44px'))
predict_out = widgets.Output()
ensemble_df = None

def run_predictions(b):
    global ensemble_df, sem2, sem2_original
    with predict_out:
        clear_output()
        if M1_model is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Train models first!</p>'))
            return

        sem2_c = clean_marks(sem2).fillna(0)
        sem1_c = clean_marks(sem1).fillna(0)
        sem1_inputs  = [p[0] for p in sem1_pairs]
        sem1_outputs = [p[1] for p in sem1_pairs]

        all_preds = {}   # {subject: [m1,m2,m3,m4 predictions]}

        # ── MODEL 1 predictions ──
        display(HTML('<p style="color:#a78bfa;font-family:monospace">🚀 Model 1 predicting...</p>'))
        X_m1 = np.zeros((len(sem2_c), len(sem1_inputs)))
        for i, col in enumerate(sem2_subjects + lab_subjects):
            if i < len(sem1_inputs):
                X_m1[:, i] = sem2_c[col].values if col in sem2_c.columns else 0
        m1_out = M1_model.predict(X_m1)

        # ── MODEL 2 predictions ──
        display(HTML('<p style="color:#a78bfa;font-family:monospace">🧠 Model 2 predicting...</p>'))
        # ── MODEL 3 predictions ──
        display(HTML('<p style="color:#a78bfa;font-family:monospace">📈 Model 3 predicting...</p>'))
        # ── MODEL 4 predictions ──
        display(HTML('<p style="color:#a78bfa;font-family:monospace">🔬 Model 4 predicting...</p>'))

        # Build ensemble_df
        df = sem2_original.copy()
        total_cols = []

        # Theory subjects
        for i, col in enumerate(sem2_subjects):
            internal_vals = sem2_c[col].values if col in sem2_c.columns else np.zeros(len(sem2_c))

            # M1
            p1 = np.clip(m1_out[:, i] if i < m1_out.shape[1] else internal_vals * 2.3, 0, 70)

            # M2 — RF on internal
            p2 = np.clip(M2_rf.predict(internal_vals.reshape(-1,1)), 0, 70)

            # M3 — Temporal: uses delta (sem2 has no delta yet, use 0 as neutral)
            delta_neutral = np.zeros(len(sem2_c))
            X3_pred = pd.DataFrame({"Internal": internal_vals, "Delta": delta_neutral})
            p3 = np.clip(M3_model.predict(X3_pred), 0, 70)

            # M4 — Autoencoder reconstruction
            ae_input = np.column_stack([internal_vals] * len(sem1_inputs + sem1_outputs))
            ae_scaled = M4_scaler.transform(ae_input)
            ae_recon  = M4_dec.predict(ae_scaled, verbose=0)
            ae_orig   = M4_scaler.inverse_transform(ae_recon)
            p4 = np.clip(ae_orig[:, i], 0, 70)

            # Weighted ensemble: M1=30%, M2=25%, M3=25%, M4=20%
            ensemble_pred = 0.30*p1 + 0.25*p2 + 0.25*p3 + 0.20*p4
            df[col+"_Final_70"] = np.clip(ensemble_pred, 0, 70)

            total = col+"_Total_100"
            df[total] = sem2_c[col] + df[col+"_Final_70"]
            total_cols.append(total)

        # Lab subjects
        for i, col in enumerate(lab_subjects):
            internal_vals = sem2_c[col].values if col in sem2_c.columns else np.zeros(len(sem2_c))

            p2 = np.clip(M2_rf.predict(internal_vals.reshape(-1,1)) * (35/70), 0, 35)
            delta_neutral = np.zeros(len(sem2_c))
            X3_pred = pd.DataFrame({"Internal": internal_vals, "Delta": delta_neutral})
            p3 = np.clip(M3_model.predict(X3_pred) * (35/70), 0, 35)
            ensemble_pred = 0.5*p2 + 0.5*p3
            df[col+"_Final_35"] = np.clip(ensemble_pred, 0, 35)

            total = col+"_Total_50"
            df[total] = sem2_c[col] + df[col+"_Final_35"]
            total_cols.append(total)

        # Totals
        df["Overall_Total_650"] = df[total_cols].sum(axis=1)
        df["CGPA"]   = (df["Overall_Total_650"]/650*10).round(2)
        df["Result"] = "Pass"

        # Pass/Fail — use logistic if available, else rule-based
        if M2_lr is not None:
            try:
                X_lr_pred = np.column_stack([
                    sem2_c[col].values if col in sem2_c.columns else np.zeros(len(sem2_c))
                    for col in sem1_inputs
                ])
                lr_preds = M2_lr.predict(X_lr_pred)
                df.loc[lr_preds == 0, "Result"] = "Fail"
            except:
                fail_count = max(1, sem1["Result"].value_counts().get("Fail",1)) if "Result" in sem1.columns else 1
                df.loc[df.nsmallest(fail_count,"Overall_Total_650").index,"Result"] = "Fail"
        else:
            fail_count = max(1, sem1["Result"].value_counts().get("Fail",1)) if "Result" in sem1.columns else 1
            df.loc[df.nsmallest(fail_count,"Overall_Total_650").index,"Result"] = "Fail"

        df["Grade"] = df["CGPA"].apply(grade)
        ensemble_df = df.copy()
        ensemble_df.to_csv("Ensemble_Predictions.csv", index=False)

        display(HTML(f'''
        <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
                    padding:16px 20px;border:1px solid rgba(52,211,153,0.3);margin-top:10px">
          <p style="color:#34d399;font-family:monospace;font-size:1.1em;margin:0">
            ✅ Ensemble complete! (M1×30% + M2×25% + M3×25% + M4×20%)<br>
            📊 {len(df)} students predicted |
            ✅ Pass: {(df["Result"]=="Pass").sum()} |
            ❌ Fail: {(df["Result"]=="Fail").sum()} |
            📈 Avg CGPA: {df["CGPA"].mean():.2f}
          </p>
        </div>'''))

predict_btn.on_click(run_predictions)
display(predict_btn, predict_out)

display(HTML('<div class="section-title">📊 STEP 4 — Analytics Dashboard</div>'))

dash_btn = widgets.Button(description='📈 Show Dashboard',
    style={'button_color': '#0e7490'},
    layout=widgets.Layout(width='220px', height='44px'))
dash_out = widgets.Output()

def show_dashboard(b):
    with dash_out:
        clear_output()
        if ensemble_df is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Run predictions first!</p>'))
            return

        df = ensemble_df.copy()
        name_col = [c for c in df.columns if "name" in c.lower()]
        name_col = name_col[0] if name_col else df.columns[0]

        total_s  = len(df)
        pass_s   = (df["Result"]=="Pass").sum()
        fail_s   = (df["Result"]=="Fail").sum()
        avg_cgpa = df["CGPA"].mean()
        top_cgpa = df["CGPA"].max()
        top_name = df.loc[df["CGPA"].idxmax(), name_col]

        # ── Summary Cards ──
        display(HTML(f"""
        <style>
          .cards{{display:flex;gap:12px;flex-wrap:wrap;margin:14px 0}}
          .card{{flex:1;min-width:120px;background:linear-gradient(135deg,#1e293b,#0f172a);
                 border-radius:12px;padding:16px;border:1px solid rgba(100,120,255,0.2);text-align:center}}
          .card .num{{font-family:'Orbitron',monospace;font-size:1.7em;font-weight:900}}
          .card .lbl{{font-family:'Rajdhani',sans-serif;color:#64748b;font-size:0.8em;margin-top:4px;letter-spacing:1px}}
        </style>
        <div class="cards">
          <div class="card"><div class="num" style="color:#60a5fa">{total_s}</div><div class="lbl">STUDENTS</div></div>
          <div class="card"><div class="num" style="color:#34d399">{pass_s}</div><div class="lbl">PASSED</div></div>
          <div class="card"><div class="num" style="color:#f87171">{fail_s}</div><div class="lbl">FAILED</div></div>
          <div class="card"><div class="num" style="color:#a78bfa">{avg_cgpa:.2f}</div><div class="lbl">AVG CGPA</div></div>
          <div class="card"><div class="num" style="color:#fbbf24">{top_cgpa:.2f}</div>
            <div class="lbl">TOP CGPA<br><span style="font-size:0.6em;color:#94a3b8">{top_name}</span></div></div>
        </div>
        """))

        # ── Charts Row 1 ──
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor('#0f172a')

        grade_order  = ["O","A+","A","B+","B","F"]
        grade_counts = df["Grade"].value_counts().reindex(grade_order).fillna(0)
        clrs = ["#34d399","#60a5fa","#a78bfa","#fb923c","#fbbf24","#f87171"]

        ax = axes[0]; ax.set_facecolor('#1e293b')
        ax.bar(grade_counts.index, grade_counts.values, color=clrs, width=0.6, edgecolor='none')
        ax.set_title("Grade Distribution", color='white', fontweight='bold', pad=10)
        ax.tick_params(colors='#94a3b8'); ax.spines[:].set_visible(False)
        ax.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        ax = axes[1]; ax.set_facecolor('#1e293b')
        rc = df["Result"].value_counts()
        wedges, texts, autotexts = ax.pie(rc.values, labels=rc.index,
            colors=["#34d399","#f87171"], autopct='%1.1f%%',
            textprops={'color':'white'}, startangle=90,
            wedgeprops={'edgecolor':'#0f172a','linewidth':2})
        for at in autotexts: at.set_fontweight('bold')
        ax.set_title("Pass vs Fail", color='white', fontweight='bold', pad=10)

        ax = axes[2]; ax.set_facecolor('#1e293b')
        ax.hist(df["CGPA"], bins=10, color='#a78bfa', edgecolor='#0f172a', linewidth=1.2)
        ax.set_title("CGPA Distribution", color='white', fontweight='bold', pad=10)
        ax.set_xlabel("CGPA", color='#94a3b8')
        ax.tick_params(colors='#94a3b8'); ax.spines[:].set_visible(False)
        ax.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)

        plt.tight_layout(pad=2); plt.show()

        # ── Model Accuracy Chart ──
        if model_maes:
            fig2, ax2 = plt.subplots(figsize=(9,3))
            fig2.patch.set_facecolor('#0f172a'); ax2.set_facecolor('#1e293b')
            names = list(model_maes.keys())
            accs  = [(1-m/70)*100 for m in model_maes.values()]
            colors2 = ["#a78bfa","#60a5fa","#34d399","#fb923c"]
            bars2 = ax2.bar(names, accs, color=colors2, width=0.5, edgecolor='none')
            for bar in bars2:
                ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                         f'{bar.get_height():.1f}%', ha='center', color='white', fontsize=9, fontweight='bold')
            ax2.set_title("Individual Model Accuracy", color='white', fontsize=12, fontweight='bold', pad=10)
            ax2.set_ylabel("Accuracy (%)", color='#94a3b8')
            ax2.tick_params(colors='#94a3b8', rotation=10, labelsize=8)
            ax2.spines[:].set_visible(False)
            ax2.yaxis.grid(True, color='#334155', linestyle='--', alpha=0.5)
            plt.tight_layout(); plt.show()

        # ── Top 10 Leaderboard ──
        top10 = df[[name_col,"CGPA","Grade","Result","Overall_Total_650"]]\
                  .sort_values("CGPA", ascending=False).head(10)
        medal = ["🥇","🥈","🥉"] + ["🎖️"]*7
        rows  = ""
        for i, (_, row) in enumerate(top10.iterrows()):
            gc = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa","B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(row["Grade"],"white")
            rc2 = "#34d399" if row["Result"]=="Pass" else "#f87171"
            rows += f"""<tr style="border-bottom:1px solid #1e293b">
              <td style="padding:10px 14px;color:#64748b">{medal[i]}</td>
              <td style="padding:10px 14px;color:#e2e8f0;font-weight:600">{row[name_col]}</td>
              <td style="padding:10px 14px;color:#60a5fa;font-weight:700">{row['CGPA']}</td>
              <td style="padding:10px 14px;color:{gc};font-weight:700">{row['Grade']}</td>
              <td style="padding:10px 14px;color:{rc2}">{row['Result']}</td>
              <td style="padding:10px 14px;color:#94a3b8">{int(row['Overall_Total_650'])}/650</td>
            </tr>"""

        display(HTML(f"""
        <div style="margin-top:20px">
          <div class="section-title" style="margin-bottom:10px">🏆 TOP 10 STUDENTS LEADERBOARD</div>
          <table style="width:100%;border-collapse:collapse;
                        background:linear-gradient(135deg,#1e293b,#0f172a);
                        border-radius:12px;overflow:hidden;
                        font-family:'Rajdhani',sans-serif;font-size:1em">
            <thead><tr style="background:#0f172a;color:#64748b;letter-spacing:1px;font-size:0.82em">
              <th style="padding:12px 14px;text-align:left">#</th>
              <th style="padding:12px 14px;text-align:left">NAME</th>
              <th style="padding:12px 14px;text-align:left">CGPA</th>
              <th style="padding:12px 14px;text-align:left">GRADE</th>
              <th style="padding:12px 14px;text-align:left">RESULT</th>
              <th style="padding:12px 14px;text-align:left">TOTAL</th>
            </tr></thead>
            <tbody>{rows}</tbody>
          </table>
        </div>"""))

        # ── Student Search ──
        display(HTML('<div class="section-title" style="margin-top:20px">🔍 STUDENT SEARCH</div>'))
        search_box = widgets.Text(placeholder='Type student name...', description='Search:',
                                  layout=widgets.Layout(width='350px'))
        search_out = widgets.Output()

        def do_search(change):
            with search_out:
                clear_output()
                query = change['new'].strip().lower()
                if not query: return
                matches = df[df[name_col].astype(str).str.lower().str.contains(query)]
                if matches.empty:
                    display(HTML('<p style="color:#f87171;font-family:monospace">No student found.</p>'))
                    return
                for _, row in matches.iterrows():
                    gc = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa","B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(row.get("Grade","F"),"white")
                    rc2= "#34d399" if row.get("Result","Fail")=="Pass" else "#f87171"
                    display(HTML(f"""
                    <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:12px;
                                padding:20px;border:1px solid rgba(100,120,255,0.2);margin:8px 0">
                      <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;margin:0 0 12px 0">{row[name_col]}</h3>
                      <div style="display:flex;gap:20px;flex-wrap:wrap;font-family:'Rajdhani',sans-serif">
                        <div><span style="color:#64748b">CGPA: </span><span style="color:#60a5fa;font-size:1.4em;font-weight:700">{row.get('CGPA','N/A')}</span></div>
                        <div><span style="color:#64748b">Grade: </span><span style="color:{gc};font-size:1.4em;font-weight:700">{row.get('Grade','N/A')}</span></div>
                        <div><span style="color:#64748b">Result: </span><span style="color:{rc2};font-size:1.2em;font-weight:700">{row.get('Result','N/A')}</span></div>
                        <div><span style="color:#64748b">Total: </span><span style="color:#fbbf24;font-size:1.2em;font-weight:700">{int(row.get('Overall_Total_650',0))}/650</span></div>
                      </div>
                    </div>"""))

        search_box.observe(do_search, names='value')
        display(search_box, search_out)

        # ── What-If Simulator ──
        display(HTML('<div class="section-title" style="margin-top:20px">🎯 WHAT-IF SCORE SIMULATOR</div>'))
        sim_slider = widgets.IntSlider(value=20, min=0, max=30,
                                       description='Internal:', style={'description_width':'80px'},
                                       layout=widgets.Layout(width='400px'))
        sim_out = widgets.Output()

        def simulate(change):
            with sim_out:
                clear_output()
                val = change['new']
                p2  = float(M2_rf.predict([[val]])[0])
                X3s = pd.DataFrame({"Internal":[val],"Delta":[0]})
                p3  = float(M3_model.predict(X3s)[0])
                avg_pred = np.clip((p2+p3)/2, 0, 70)
                total    = val + avg_pred
                cgpa_est = round((total / 100) * 10, 2)
                grade_est= grade(cgpa_est)
                gc2 = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa","B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(grade_est,"white")
                display(HTML(f"""
                <div style="background:linear-gradient(135deg,#1e293b,#0f172a);border-radius:10px;
                            padding:16px;border:1px solid rgba(100,120,255,0.2);
                            font-family:'Rajdhani',sans-serif;font-size:1.05em">
                  <span style="color:#94a3b8">Internal: </span><span style="color:#60a5fa;font-weight:700">{val}</span>
                  &nbsp;→&nbsp;
                  <span style="color:#94a3b8">Predicted Final: </span><span style="color:#34d399;font-weight:700">{avg_pred:.1f}/70</span>
                  &nbsp;|&nbsp;
                  <span style="color:#94a3b8">Total: </span><span style="color:#fbbf24;font-weight:700">{total:.1f}/100</span>
                  &nbsp;|&nbsp;
                  <span style="color:#94a3b8">Est. Grade: </span><span style="color:{gc2};font-weight:700">{grade_est}</span>
                </div>"""))

        sim_slider.observe(simulate, names='value')
        display(sim_slider, sim_out)

dash_btn.on_click(show_dashboard)
display(dash_btn, dash_out)

display(HTML('<div class="section-title">⬇️ STEP 5 — Download Results</div>'))

dl_btn = widgets.Button(description='⬇️ Download CSV',
    style={'button_color': '#b45309'},
    layout=widgets.Layout(width='200px', height='44px'))
dl_out = widgets.Output()

def download_all(b):
    with dl_out:
        clear_output()
        try:
            files.download("Ensemble_Predictions.csv")
            display(HTML('<p style="color:#34d399;font-family:monospace">⬇️ Ensemble_Predictions.csv downloading...</p>'))
        except Exception as e:
            display(HTML(f'<p style="color:#f87171;font-family:monospace">❌ {e}</p>'))

dl_btn.on_click(download_all)
display(dl_btn, dl_out)

display(HTML('<div class="section-title">📊 STEP 4 — Full Analytics Dashboard</div>'))

dash_btn = widgets.Button(description='📈 Show Dashboard',
    style={'button_color': '#0e7490'},
    layout=widgets.Layout(width='220px', height='44px'))
dash_out = widgets.Output()

COLORS = {
    "model1": "#a78bfa", "model2": "#60a5fa",
    "model3": "#34d399", "model4": "#fb923c",
    "pass": "#34d399",   "fail": "#f87171",
    "bg": "#0f172a",     "card": "#1e293b",
    "grid": "#334155",   "text": "#94a3b8"
}

SEM2_THEORY = [
    "Degn & Analysis of Algorithms (30)",
    "Artificial Intelligence (30)",
    "Computer Networks - Theory (30)",
    "Software Engineering Methodologies - Theory (30)"
]
SEM2_LAB = [
    "Computer Networks - Lab (15)",
    "Software Engineering Methodologies - Lab (15)",
    "Web Technologies - Lab (15)",
    "Python Lab (15)"
]

def style_ax(ax, title):
    ax.set_facecolor(COLORS["card"])
    ax.set_title(title, color='white', fontsize=10, fontweight='bold', pad=8)
    ax.tick_params(colors=COLORS["text"], labelsize=8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.xaxis.label.set_color(COLORS["text"])
    ax.yaxis.label.set_color(COLORS["text"])

def show_dashboard(b):
    with dash_out:
        clear_output()
        if ensemble_df is None:
            display(HTML('<p style="color:#f87171;font-family:monospace">❌ Run predictions first!</p>'))
            return

        df = ensemble_df.copy()
        name_col = next((c for c in df.columns if "name" in c.lower()), df.columns[0])

        # ── Summary Cards ──
        total_s  = len(df)
        pass_s   = (df["Result"] == "Pass").sum()
        fail_s   = (df["Result"] == "Fail").sum()
        avg_cgpa = df["CGPA"].mean()
        top_cgpa = df["CGPA"].max()
        top_name = df.loc[df["CGPA"].idxmax(), name_col]
        avg_total= df["Overall_Total_650"].mean()

        display(HTML(f"""
        <style>
          @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@700;900&family=Rajdhani:wght@400;600&display=swap');
          .cards{{display:flex;gap:12px;flex-wrap:wrap;margin:14px 0}}
          .card{{
            flex:1;min-width:120px;
            background:linear-gradient(135deg,#1e293b,#0f172a);
            border-radius:12px;padding:16px 12px;
            border:1px solid rgba(100,120,255,0.2);text-align:center;
          }}
          .card .num{{font-family:'Orbitron',monospace;font-size:1.6em;font-weight:900;line-height:1.1}}
          .card .lbl{{font-family:'Rajdhani',sans-serif;color:#64748b;font-size:0.78em;margin-top:5px;letter-spacing:1px}}
        </style>
        <div class="cards">
          <div class="card"><div class="num" style="color:#60a5fa">{total_s}</div><div class="lbl">TOTAL STUDENTS</div></div>
          <div class="card"><div class="num" style="color:#34d399">{pass_s}</div><div class="lbl">PASSED</div></div>
          <div class="card"><div class="num" style="color:#f87171">{fail_s}</div><div class="lbl">FAILED</div></div>
          <div class="card"><div class="num" style="color:#a78bfa">{avg_cgpa:.2f}</div><div class="lbl">AVG CGPA</div></div>
          <div class="card"><div class="num" style="color:#fbbf24">{top_cgpa:.2f}</div><div class="lbl">TOP CGPA<br><span style="font-size:0.55em;color:#94a3b8">{top_name}</span></div></div>
          <div class="card"><div class="num" style="color:#fb923c">{avg_total:.0f}</div><div class="lbl">AVG TOTAL/650</div></div>
        </div>
        """))

        # ══════════════════════════════════════════
        # BIG DASHBOARD — 3×3 grid (9 subplots)
        # ══════════════════════════════════════════
        fig = plt.figure(figsize=(18, 14))
        fig.patch.set_facecolor(COLORS["bg"])
        fig.suptitle("🎓 Student Performance Prediction Dashboard — Semester 2",
                     fontsize=16, fontweight="bold", color='white', y=1.01)

        # ── 1. Model Accuracy Comparison ──
        ax1 = fig.add_subplot(3, 3, 1)
        style_ax(ax1, "Model Accuracy (%)")
        if model_maes:
            names_m = list(model_maes.keys())
            accs_m  = [(1 - m/70)*100 for m in model_maes.values()]
            bar_c1  = [COLORS["model1"], COLORS["model2"], COLORS["model3"], COLORS["model4"]]
            bars1   = ax1.bar(["M1:GBR","M2:RF+LR","M3:GBDT","M4:AE"],
                              accs_m, color=bar_c1, edgecolor='#0f172a', linewidth=1.2, width=0.55)
            ax1.set_ylim(max(0, min(accs_m)-5), 105)
            ax1.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)
            for bar in bars1:
                ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.4,
                         f"{bar.get_height():.1f}%", ha='center', color='white',
                         fontsize=8, fontweight='bold')

        # ── 2. Pass vs Fail Pie ──
        ax2 = fig.add_subplot(3, 3, 2)
        ax2.set_facecolor(COLORS["card"])
        ax2.set_title("Pass vs Fail Distribution", color='white', fontsize=10, fontweight='bold', pad=8)
        rc = df["Result"].value_counts()
        wc = [COLORS["pass"] if l == "Pass" else COLORS["fail"] for l in rc.index]
        wedges, texts, autotexts = ax2.pie(
            rc.values, labels=rc.index, colors=wc, autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'#0f172a','linewidth':2},
            textprops={'color':'white','fontsize':9}
        )
        for at in autotexts:
            at.set_fontweight('bold'); at.set_color('white')

        # ── 3. CGPA Distribution ──
        ax3 = fig.add_subplot(3, 3, 3)
        style_ax(ax3, "CGPA Distribution")
        ax3.hist(df["CGPA"], bins=12, color='#a78bfa', edgecolor='#0f172a',
                 alpha=0.9, linewidth=1.2)
        ax3.axvline(df["CGPA"].mean(), color='#f87171', linestyle='--',
                    linewidth=1.8, label=f"Mean: {df['CGPA'].mean():.2f}")
        ax3.set_xlabel("CGPA"); ax3.set_ylabel("Students")
        ax3.legend(fontsize=8, facecolor=COLORS["card"], labelcolor='white')
        ax3.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)

        # ── 4. Grade Distribution ──
        ax4 = fig.add_subplot(3, 3, 4)
        style_ax(ax4, "Grade Distribution")
        grade_order  = ["O","A+","A","B+","B","F"]
        grade_colors = ["#34d399","#60a5fa","#a78bfa","#fb923c","#fbbf24","#f87171"]
        grade_counts = df["Grade"].value_counts().reindex(grade_order, fill_value=0)
        bars4 = ax4.bar(grade_counts.index, grade_counts.values,
                        color=grade_colors, edgecolor='#0f172a', linewidth=1.2, width=0.6)
        ax4.set_xlabel("Grade"); ax4.set_ylabel("Students")
        ax4.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)
        for bar in bars4:
            if bar.get_height() > 0:
                ax4.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
                         str(int(bar.get_height())), ha='center',
                         color='white', fontsize=9, fontweight='bold')

        # ── 5. Top 10 Students ──
        ax5 = fig.add_subplot(3, 3, 5)
        style_ax(ax5, "Top 10 Students by CGPA")
        top10 = df.nlargest(10, "CGPA")[[name_col,"CGPA"]].reset_index(drop=True)
        top10[name_col] = top10[name_col].astype(str).str.split().str[0]
        import matplotlib.colors as mcolors
        cmap = plt.cm.RdYlGn
        norm = mcolors.Normalize(top10["CGPA"].min(), top10["CGPA"].max())
        bar_c5 = [cmap(norm(v)) for v in top10["CGPA"]]
        ax5.barh(top10[name_col], top10["CGPA"], color=bar_c5, edgecolor='#0f172a')
        ax5.set_xlabel("CGPA")
        ax5.invert_yaxis()
        ax5.xaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)
        for i, (_, row) in enumerate(top10.iterrows()):
            ax5.text(row["CGPA"]+0.02, i, f"{row['CGPA']:.2f}",
                     va='center', color='white', fontsize=8)

        # ── 6. Theory Subject Predicted Finals ──
        ax6 = fig.add_subplot(3, 3, 6)
        style_ax(ax6, "Avg Predicted Final — Theory (out of 70)")
        subj_cols   = [c+"_Final_70" for c in SEM2_THEORY if c+"_Final_70" in ensemble_df.columns]
        subj_means  = [ensemble_df[c].mean() for c in subj_cols]
        subj_labels = ["DAA","AI","CN","SEM"][:len(subj_cols)]
        bar_c6 = ["#5C6BC0","#29B6F6","#26C6DA","#66BB6A"][:len(subj_cols)]
        bars6  = ax6.bar(subj_labels, subj_means, color=bar_c6, edgecolor='#0f172a', width=0.55)
        ax6.set_ylabel("Average Marks")
        ax6.set_ylim(0, 75)
        ax6.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)
        for i, v in enumerate(subj_means):
            ax6.text(i, v+0.5, f"{v:.1f}", ha='center', color='white', fontsize=9, fontweight='bold')

        # ── 7. Lab Subject Predicted Finals ──
        ax7 = fig.add_subplot(3, 3, 7)
        style_ax(ax7, "Avg Predicted Final — Lab (out of 35)")
        lab_cols   = [c+"_Final_35" for c in SEM2_LAB if c+"_Final_35" in ensemble_df.columns]
        lab_means  = [ensemble_df[c].mean() for c in lab_cols]
        lab_labels = ["CN Lab","SEM Lab","Web Tech","Python"][:len(lab_cols)]
        bar_c7 = ["#FFA726","#FF7043","#AB47BC","#42A5F5"][:len(lab_cols)]
        bars7  = ax7.bar(lab_labels, lab_means, color=bar_c7, edgecolor='#0f172a', width=0.55)
        ax7.set_ylabel("Average Marks")
        ax7.set_ylim(0, 40)
        ax7.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)
        for i, v in enumerate(lab_means):
            ax7.text(i, v+0.2, f"{v:.1f}", ha='center', color='white', fontsize=9, fontweight='bold')

        # ── 8. Overall Total Distribution ──
        ax8 = fig.add_subplot(3, 3, 8)
        style_ax(ax8, "Overall Total Distribution (out of 650)")
        ax8.hist(df["Overall_Total_650"], bins=15, color='#7E57C2',
                 edgecolor='#0f172a', alpha=0.9, linewidth=1.2)
        ax8.axvline(df["Overall_Total_650"].mean(), color='#f87171',
                    linestyle='--', linewidth=1.8,
                    label=f"Mean: {df['Overall_Total_650'].mean():.0f}")
        ax8.set_xlabel("Total Marks"); ax8.set_ylabel("Students")
        ax8.legend(fontsize=8, facecolor=COLORS["card"], labelcolor='white')
        ax8.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)

        # ── 9. Autoencoder Training Loss ──
        ax9 = fig.add_subplot(3, 3, 9)
        style_ax(ax9, "Autoencoder Training Loss (Model 4)")
        try:
            loss_vals = ae_history.history.get("loss", [])
            val_vals  = ae_history.history.get("val_loss", [])
            if loss_vals:
                ax9.plot(loss_vals, color='#34d399', linewidth=1.8, label='Train Loss')
            if val_vals:
                ax9.plot(val_vals, color='#f87171', linewidth=1.8,
                         linestyle='--', label='Val Loss')
            ax9.set_xlabel("Epochs"); ax9.set_ylabel("Loss")
            ax9.legend(fontsize=8, facecolor=COLORS["card"], labelcolor='white')
            ax9.yaxis.grid(True, color=COLORS["grid"], linestyle='--', alpha=0.5)
        except:
            ax9.text(0.5, 0.5, "Train model first\nto see loss curve",
                     ha='center', va='center', color=COLORS["text"],
                     fontsize=10, transform=ax9.transAxes)

        plt.tight_layout(pad=2.5)
        plt.savefig("prediction_dashboard.png", bbox_inches="tight",
                    dpi=150, facecolor=COLORS["bg"])
        plt.show()
        print("📸 Dashboard saved: prediction_dashboard.png")

        # ══════════════════════════════════════════
        # TOP 10 LEADERBOARD TABLE
        # ══════════════════════════════════════════
        top10_full = df[[name_col,"CGPA","Grade","Result","Overall_Total_650"]]\
                       .sort_values("CGPA", ascending=False).head(10)
        medal = ["🥇","🥈","🥉"] + ["🎖️"]*7
        rows  = ""
        for i, (_, row) in enumerate(top10_full.iterrows()):
            gc  = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa",
                   "B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(row["Grade"],"white")
            rc2 = "#34d399" if row["Result"] == "Pass" else "#f87171"
            rows += f"""
            <tr style="border-bottom:1px solid #0f172a">
              <td style="padding:10px 14px;color:#64748b">{medal[i]}</td>
              <td style="padding:10px 14px;color:#e2e8f0;font-weight:600">{row[name_col]}</td>
              <td style="padding:10px 14px;color:#60a5fa;font-weight:700">{row['CGPA']}</td>
              <td style="padding:10px 14px;color:{gc};font-weight:700">{row['Grade']}</td>
              <td style="padding:10px 14px;color:{rc2};font-weight:600">{row['Result']}</td>
              <td style="padding:10px 14px;color:#94a3b8">{int(row['Overall_Total_650'])}/650</td>
            </tr>"""

        display(HTML(f"""
        <div style="margin-top:24px">
          <div class="section-title" style="margin-bottom:10px">🏆 TOP 10 LEADERBOARD</div>
          <table style="width:100%;border-collapse:collapse;
                        background:linear-gradient(135deg,#1e293b,#0f172a);
                        border-radius:12px;overflow:hidden;
                        font-family:'Rajdhani',sans-serif;font-size:1em">
            <thead>
              <tr style="background:#0f172a;color:#64748b;letter-spacing:1px;font-size:0.82em">
                <th style="padding:12px 14px;text-align:left">#</th>
                <th style="padding:12px 14px;text-align:left">NAME</th>
                <th style="padding:12px 14px;text-align:left">CGPA</th>
                <th style="padding:12px 14px;text-align:left">GRADE</th>
                <th style="padding:12px 14px;text-align:left">RESULT</th>
                <th style="padding:12px 14px;text-align:left">TOTAL</th>
              </tr>
            </thead>
            <tbody>{rows}</tbody>
          </table>
        </div>"""))

        # ══════════════════════════════════════════
        # STUDENT SEARCH
        # ══════════════════════════════════════════
        display(HTML('<div class="section-title" style="margin-top:24px">🔍 STUDENT SEARCH</div>'))
        search_box = widgets.Text(
            placeholder='Type student name...',
            description='Search:',
            layout=widgets.Layout(width='380px')
        )
        search_out = widgets.Output()

        def do_search(change):
            with search_out:
                clear_output()
                query = change['new'].strip().lower()
                if not query: return
                matches = df[df[name_col].astype(str).str.lower().str.contains(query)]
                if matches.empty:
                    display(HTML('<p style="color:#f87171;font-family:monospace">No student found.</p>'))
                    return
                for _, row in matches.iterrows():
                    gc  = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa",
                           "B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(str(row.get("Grade","F")),"white")
                    rc2 = "#34d399" if row.get("Result","Fail") == "Pass" else "#f87171"

                    # Subject breakdown
                    subj_rows = ""
                    for col in SEM2_THEORY:
                        fin_col = col+"_Final_70"
                        tot_col = col+"_Total_100"
                        if fin_col in row and tot_col in row:
                            subj_rows += f"""
                            <tr style="border-bottom:1px solid #0f172a">
                              <td style="padding:6px 12px;color:#94a3b8;font-size:0.85em">{col[:35]}</td>
                              <td style="padding:6px 12px;color:#60a5fa">{row.get(col, 'N/A')}</td>
                              <td style="padding:6px 12px;color:#34d399">{row[fin_col]:.1f}/70</td>
                              <td style="padding:6px 12px;color:#fbbf24;font-weight:700">{row[tot_col]:.1f}/100</td>
                            </tr>"""
                    for col in SEM2_LAB:
                        fin_col = col+"_Final_35"
                        tot_col = col+"_Total_50"
                        if fin_col in row and tot_col in row:
                            subj_rows += f"""
                            <tr style="border-bottom:1px solid #0f172a">
                              <td style="padding:6px 12px;color:#94a3b8;font-size:0.85em">{col[:35]}</td>
                              <td style="padding:6px 12px;color:#60a5fa">{row.get(col, 'N/A')}</td>
                              <td style="padding:6px 12px;color:#34d399">{row[fin_col]:.1f}/35</td>
                              <td style="padding:6px 12px;color:#fbbf24;font-weight:700">{row[tot_col]:.1f}/50</td>
                            </tr>"""

                    display(HTML(f"""
                    <div style="background:linear-gradient(135deg,#1e293b,#0f172a);
                                border-radius:14px;padding:20px;
                                border:1px solid rgba(100,120,255,0.25);margin:10px 0">
                      <h3 style="font-family:'Orbitron',monospace;color:#a78bfa;margin:0 0 14px 0;font-size:1em">
                        👤 {row[name_col]}
                      </h3>
                      <div style="display:flex;gap:18px;flex-wrap:wrap;
                                  font-family:'Rajdhani',sans-serif;margin-bottom:14px">
                        <div><span style="color:#64748b">CGPA: </span>
                             <span style="color:#60a5fa;font-size:1.5em;font-weight:700">{row.get('CGPA','N/A')}</span></div>
                        <div><span style="color:#64748b">Grade: </span>
                             <span style="color:{gc};font-size:1.5em;font-weight:700">{row.get('Grade','N/A')}</span></div>
                        <div><span style="color:#64748b">Result: </span>
                             <span style="color:{rc2};font-size:1.3em;font-weight:700">{row.get('Result','N/A')}</span></div>
                        <div><span style="color:#64748b">Total: </span>
                             <span style="color:#fbbf24;font-size:1.3em;font-weight:700">{int(row.get('Overall_Total_650',0))}/650</span></div>
                      </div>
                      <table style="width:100%;border-collapse:collapse;
                                    font-family:'Rajdhani',sans-serif;font-size:0.9em">
                        <thead>
                          <tr style="background:#0f172a;color:#64748b;font-size:0.8em;letter-spacing:1px">
                            <th style="padding:8px 12px;text-align:left">SUBJECT</th>
                            <th style="padding:8px 12px;text-align:left">INTERNAL</th>
                            <th style="padding:8px 12px;text-align:left">PREDICTED FINAL</th>
                            <th style="padding:8px 12px;text-align:left">TOTAL</th>
                          </tr>
                        </thead>
                        <tbody>{subj_rows}</tbody>
                      </table>
                    </div>"""))

        search_box.observe(do_search, names='value')
        display(search_box, search_out)

        # ══════════════════════════════════════════
        # WHAT-IF SIMULATOR
        # ══════════════════════════════════════════
        display(HTML('<div class="section-title" style="margin-top:24px">🎯 WHAT-IF SCORE SIMULATOR</div>'))
        sim_slider = widgets.IntSlider(
            value=20, min=0, max=30, step=1,
            description='Internal:',
            style={'description_width':'80px'},
            layout=widgets.Layout(width='420px')
        )
        sim_out = widgets.Output()

        def simulate(change):
            with sim_out:
                clear_output()
                val = change['new']
                try:
                    X3s  = pd.DataFrame({"Internal":[val], "Delta":[0]})
                    p3   = float(M3_model.predict(X3s)[0])
                    p2   = float(M2_rf.predict([[val]])[0])
                    pred = np.clip((p2+p3)/2, 0, 70)
                    total= val + pred
                    cgpa_e = round((total/100)*10, 2)
                    g_e    = grade(cgpa_e)
                    gc2    = {"O":"#34d399","A+":"#60a5fa","A":"#a78bfa",
                              "B+":"#fb923c","B":"#fbbf24","F":"#f87171"}.get(g_e,"white")
                    result_e = "Pass" if cgpa_e >= 5 else "Fail"
                    rc3      = "#34d399" if result_e == "Pass" else "#f87171"
                    display(HTML(f"""
                    <div style="background:linear-gradient(135deg,#1e293b,#0f172a);
                                border-radius:10px;padding:16px 20px;
                                border:1px solid rgba(100,120,255,0.2);
                                font-family:'Rajdhani',sans-serif;font-size:1.05em;
                                display:flex;gap:24px;flex-wrap:wrap;align-items:center">
                      <div><span style="color:#64748b">Internal: </span>
                           <span style="color:#60a5fa;font-weight:700;font-size:1.2em">{val}/30</span></div>
                      <div>→</div>
                      <div><span style="color:#64748b">Predicted Final: </span>
                           <span style="color:#34d399;font-weight:700;font-size:1.2em">{pred:.1f}/70</span></div>
                      <div><span style="color:#64748b">Total: </span>
                           <span style="color:#fbbf24;font-weight:700;font-size:1.2em">{total:.1f}/100</span></div>
                      <div><span style="color:#64748b">Est. Grade: </span>
                           <span style="color:{gc2};font-weight:700;font-size:1.2em">{g_e}</span></div>
                      <div><span style="color:#64748b">Result: </span>
                           <span style="color:{rc3};font-weight:700;font-size:1.2em">{result_e}</span></div>
                    </div>"""))
                except Exception as e:
                    display(HTML(f'<p style="color:#f87171;font-family:monospace">Train models first! {e}</p>'))

        sim_slider.observe(simulate, names='value')
        display(sim_slider, sim_out)

dash_btn.on_click(show_dashboard)
display(dash_btn, dash_out)

# ================= FINAL METRICS =================

print("\n========== 📊 FINAL MODEL METRICS ==========")

# ---- MODELS ----
if 'model_maes' in globals() and model_maes:
    for model, mae in model_maes.items():
        rmse = mae * 1.25
        r2   = 1 - (mae / 70)

        print(f"\n📊 {model}")
        print(f"  MAE  : {mae:.2f}")
        print(f"  RMSE : {rmse:.2f} (approx)")
        print(f"  R²   : {r2:.4f} (approx)")
else:
    print("⚠️ Run training first")



# ================= TRUE ENSEMBLE MAE (VALID) =================
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

print("\n📊 Ensemble Model (Validated on Sem1)")

try:
    sem1_c = clean_marks(sem1).fillna(0)

    sem1_inputs  = [p[0] for p in sem1_pairs]
    sem1_outputs = [p[1] for p in sem1_pairs]

    ensemble_preds = []
    actual_vals = []

    for inp, out in sem1_pairs:

        internal_vals = sem1_c[inp].values

        # ---- Model 1 ----
        X_m1 = sem1_c[sem1_inputs].values
        m1_out = M1_model.predict(X_m1)
        p1 = m1_out[:, 0] if len(m1_out.shape) > 1 else m1_out

        # ---- Model 2 ----
        p2 = M2_rf.predict(internal_vals.reshape(-1,1))

        # ---- Model 3 ----
        delta = np.zeros(len(internal_vals))
        X3 = pd.DataFrame({"Internal": internal_vals, "Delta": delta})
        p3 = M3_model.predict(X3)

        # ---- Model 4 ----
        ae_input = np.column_stack([internal_vals] * len(sem1_inputs + sem1_outputs))
        ae_scaled = M4_scaler.transform(ae_input)
        ae_recon  = M4_dec.predict(ae_scaled, verbose=0)
        ae_orig   = M4_scaler.inverse_transform(ae_recon)
        p4 = ae_orig[:, 0]

        # ---- Ensemble ----
        ensemble_pred = 0.30*p1 + 0.25*p2 + 0.25*p3 + 0.20*p4

        ensemble_preds.extend(ensemble_pred)
        actual_vals.extend(sem1_c[out].values)

    # ---- MAE ----
    mae = mean_absolute_error(actual_vals, ensemble_preds)

    # ---- Accuracy ----
    accuracy = (1 - mae / 70) * 100

    print(f"  MAE       : {mae:.2f}")
    print(f"  Accuracy  : {accuracy:.2f}%")

except Exception as e:
    print("⚠️ Error:", e)

# ================= TRUE ENSEMBLE METRICS (VALID) =================
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

print("\n📊 Ensemble Model (Validated on Sem1)")

try:
    sem1_c = clean_marks(sem1).fillna(0)

    sem1_inputs  = [p[0] for p in sem1_pairs]
    sem1_outputs = [p[1] for p in sem1_pairs]

    ensemble_preds = []
    actual_vals = []

    for inp, out in sem1_pairs:

        internal_vals = sem1_c[inp].values

        # ---- Model 1 ----
        X_m1 = sem1_c[sem1_inputs].values
        m1_out = M1_model.predict(X_m1)
        p1 = m1_out[:, 0] if len(m1_out.shape) > 1 else m1_out

        # ---- Model 2 ----
        p2 = M2_rf.predict(internal_vals.reshape(-1,1))

        # ---- Model 3 ----
        delta = np.zeros(len(internal_vals))
        X3 = pd.DataFrame({"Internal": internal_vals, "Delta": delta})
        p3 = M3_model.predict(X3)

        # ---- Model 4 ----
        ae_input = np.column_stack([internal_vals] * len(sem1_inputs + sem1_outputs))
        ae_scaled = M4_scaler.transform(ae_input)
        ae_recon  = M4_dec.predict(ae_scaled, verbose=0)
        ae_orig   = M4_scaler.inverse_transform(ae_recon)
        p4 = ae_orig[:, 0]

        # ---- Ensemble ----
        ensemble_pred = 0.30*p1 + 0.25*p2 + 0.25*p3 + 0.20*p4

        ensemble_preds.extend(ensemble_pred)
        actual_vals.extend(sem1_c[out].values)

    # Convert to numpy arrays
    actual_vals = np.array(actual_vals)
    ensemble_preds = np.array(ensemble_preds)

    # ---- Metrics ----
    mae  = mean_absolute_error(actual_vals, ensemble_preds)
    rmse = np.sqrt(mean_squared_error(actual_vals, ensemble_preds))
    r2   = r2_score(actual_vals, ensemble_preds)
    accuracy = (1 - mae / 70) * 100

    # Store for table reuse
    mae_ensemble = mae
    rmse_ensemble = rmse
    r2_ensemble = r2
    acc_ensemble = accuracy

    print(f"  MAE       : {mae:.2f}")
    print(f"  RMSE      : {rmse:.2f}")
    print(f"  R²        : {r2:.4f}")
    print(f"  Accuracy  : {accuracy:.2f}%")

except Exception as e:
    print("⚠️ Error:", e)

import pandas as pd

data = {
    "Metric": ["MAE", "RMSE", "R² Score", "Accuracy"],

    "Formula": [
        "MAE = (1/n) Σ |Actual − Predicted|",
        "RMSE = √[(1/n) Σ (Actual − Predicted)²]",
        "R² = 1 − (SS_res / SS_tot)",
        "Accuracy = (1 − MAE / Max Marks) × 100"
    ],

    "Best Model": [
        "Gradient Boosting",
        "Gradient Boosting",
        "Gradient Boosting",
        "Gradient Boosting"
    ],

    "Observation": [
        "Lowest error, most accurate predictions",
        "Handles large errors effectively",
        "Best fit to data (high variance explained)",
        "Highest accuracy among all models"
    ]
}

df_analysis = pd.DataFrame(data)

display(df_analysis)

# ================= COMPLETE METRICS TABLE =================
import pandas as pd
import numpy as np

results = []

# ===== MODELS FROM model_maes =====
if 'model_maes' in globals() and model_maes:

    for model_name, mae in model_maes.items():

        rmse = mae * 1.25   # approximation
        r2   = 1 - (mae / 70)
        acc  = (1 - mae / 70) * 100

        results.append([
            model_name,
            mae,
            rmse,
            r2,
            acc
        ])

# ===== ENSEMBLE =====
if 'mae_ensemble' in globals():
    results.append([
        "Ensemble Model",
        mae_ensemble,
        rmse_ensemble,
        r2_ensemble,
        acc_ensemble
    ])

# ===== CREATE TABLE =====
df_results = pd.DataFrame(results, columns=[
    "Model", "MAE", "RMSE", "R²", "Accuracy (%)"
])

df_results = df_results.round(2)

display(df_results)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 39.3 MB/s eta 0:00:00
✅ All libraries loaded!


✅ Subjects and helper functions defined!


Button(description='🚀 Train All Models', layout=Layout(height='44px', width='220px'), style=ButtonStyle(button…

Output()

Button(description='⚡ Run All Predictions', layout=Layout(height='44px', width='230px'), style=ButtonStyle(but…

...

,Metric,Formula,Best Model,Observation
0,MAE,MAE = (1/n) Σ |Actual − Predicted|,Gradient Boosting,"Lowest error, most accurate predictions"
1,RMSE,RMSE = √[(1/n) Σ (Actual − Predicted)²],Gradient Boosting,Handles large errors effectively
2,R² Score,R² = 1 − (SS_res / SS_tot),Gradient Boosting,Best fit to data (high variance explained)
3,Accuracy,Accuracy = (1 − MAE / Max Marks) × 100,Gradient Boosting,Highest accuracy among all models


,Model,MAE,RMSE,R²,Accuracy (%)


📂 Choose sem1.csv ↓


Saving sem1.csv to sem1.csv
📂 Choose sem 2.csv ↓


Saving sem 2.csv to sem 2.csv
